In [1]:
import torch
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import os

from polar_bimamba_dataset import BER, FER, PolarCodeDataset, EbN0_to_std
from polar_bimamba_model import PolarBiMambaDecoder
from polar_bimamba_init import load_checkpoint


/home/aayush/Desktop/5G-Polar/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

experiment_path = "./bimamba_results/C538BDB2743E800897847A061F425C9C"
TEST_BATCH_SIZE = 32
SNRs = [3, 4, 5, 6,10]  # test


checkpoint = load_checkpoint(experiment_path)
config = checkpoint['config']

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PolarBiMambaDecoder(config).to(device)


In [3]:
best_model_path = os.path.join(experiment_path, "best_model.pt")
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"Loaded best model from {best_model_path}")
else:
    model.load_state_dict(checkpoint['model'])
    print("Loaded last saved model from checkpoint")



Loaded best model from ./bimamba_results/C538BDB2743E800897847A061F425C9C/best_model.pt


In [4]:
std_test = [EbN0_to_std(snr, config.code.k / config.code.n) for snr in SNRs]

test_loader_list = []
for sigma in std_test:
    test_dataset = PolarCodeDataset(
        code=config.code,
        sigma_list=[sigma],
        dataset_size=TEST_BATCH_SIZE * 1000,  
        zero_cw=False
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=TEST_BATCH_SIZE,
        shuffle=False,
        num_workers=0  # for notebook
    )
    test_loader_list.append(test_loader)


In [5]:
model.eval()
results = {}

with torch.no_grad():
    for snr, test_loader in zip(SNRs, test_loader_list):
        total_ber = 0.0
        total_fer = 0.0
        total_samples = 0
        
        for m, x, z, y, magnitude, syndrome in tqdm(test_loader, desc=f"Testing SNR {snr} dB"):
            z_pred = model(magnitude.to(device), syndrome.to(device))
            x_pred = model.get_codeword(z_pred, y.to(device))
            
            batch_size = x.shape[0]
            total_ber += BER(x_pred, x.to(device)) * batch_size
            total_fer += FER(x_pred, x.to(device)) * batch_size
            total_samples += batch_size
        
        total_ber /= total_samples
        total_fer /= total_samples
        results[f"BER_{snr}dB"] = total_ber
        results[f"FER_{snr}dB"] = total_fer
        
        print(f"SNR={snr} dB: BER={total_ber:.2e}, FER={total_fer:.2e}")

print("\nTest results for all SNRs:")
print(results)

Testing SNR 3 dB: 100%|██████████| 1000/1000 [00:43<00:00, 23.21it/s]


SNR=3 dB: BER=2.58e-02, FER=4.07e-01


Testing SNR 4 dB: 100%|██████████| 1000/1000 [00:41<00:00, 23.87it/s]


SNR=4 dB: BER=1.43e-02, FER=2.54e-01


Testing SNR 5 dB: 100%|██████████| 1000/1000 [00:34<00:00, 29.40it/s]


SNR=5 dB: BER=6.93e-03, FER=1.31e-01


Testing SNR 6 dB: 100%|██████████| 1000/1000 [00:21<00:00, 45.66it/s]


SNR=6 dB: BER=2.73e-03, FER=5.42e-02


Testing SNR 10 dB: 100%|██████████| 1000/1000 [00:06<00:00, 150.56it/s]

SNR=10 dB: BER=9.77e-06, FER=2.19e-04

Test results for all SNRs:
{'BER_3dB': 0.025841796875, 'FER_3dB': 0.40690625, 'BER_4dB': 0.0143369140625, 'FER_4dB': 0.25390625, 'BER_5dB': 0.0069326171875, 'FER_5dB': 0.1306875, 'BER_6dB': 0.0027265625, 'FER_6dB': 0.05421875, 'BER_10dB': 9.765625e-06, 'FER_10dB': 0.00021875}
